# TSL-51 Training

Train Thai Sign Language recognition models on Google Colab.

## 1. Setup (Run 01_setup.ipynb first)

In [ ]:
import os
import sys
import torch
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/TSL'
os.chdir(WORK_DIR)
sys.path.insert(0, os.path.join(WORK_DIR, 'src'))

print(f'Working directory: {os.getcwd()}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 2. Training Configuration

In [ ]:
# Training parameters
CONFIG = {
    # Dataset
    'dataset': 'tsl51_user_sign',    # Options: tsl51_user_sign, tsl51_expert, tsl51_expert_full, tsl51_combined
    'seq_mode': True,                 # True = use per-frame sequences (T=30), False = mean-aggregated
    'target_frames': 30,              # Fixed sequence length for seq_mode

    # Model
    'model_type': 'gru',             # Options: gru, mlp
    'hidden_dim': 256,
    'num_layers': 3,
    'dropout': 0.3,

    # Training
    'learning_rate': 0.001,
    'batch_size': 64,
    'epochs': 100,
    'k_folds': 5,
    'patience': 15,
    'test_split': 0.2,

    # Augmentation (seq_mode-aware: noise, scale, flip, noise_scale)
    'augment_factor': 5,
    'noise_level': 0.01,
    'scale_range': (0.95, 1.05),
}

print('Training configuration:')
for key, value in CONFIG.items():
    print(f'  {key}: {value}')

## 3. Load Dataset

When ``seq_mode=True``, loads per-frame sequences (N, T, 162) instead of mean-aggregated (N, 162).

In [ ]:
from src.data.loader import (
    load_tsl51_user_sign,
    load_tsl51_user_sign_sequence,
    load_tsl51_expert,
    load_tsl51_combined,
    load_tsl51_expert_full,
)
from src.train.augment import augment_data

seq_mode = CONFIG.get('seq_mode', False)
target_frames = CONFIG.get('target_frames', 30)

if seq_mode and CONFIG['dataset'] == 'tsl51_user_sign':
    print(f'Loading user_sign as sequences (T={target_frames})...')
    X, y, classes = load_tsl51_user_sign_sequence(target_frames=target_frames)
    print(f'Shape: {X.shape}  (N, T={target_frames}, F=162)')
else:
    dataset_loaders = {
        'tsl51_user_sign': load_tsl51_user_sign,
        'tsl51_expert': lambda: load_tsl51_expert(include_augmented=False),
        'tsl51_expert_full': load_tsl51_expert_full,
        'tsl51_combined': load_tsl51_combined,
    }
    print(f'Loading dataset: {CONFIG["dataset"]}')
    loader_func = dataset_loaders.get(CONFIG['dataset'], load_tsl51_user_sign)
    X, y, classes = loader_func()
    print(f'Shape: {X.shape}  (N, F)')

print(f'Total samples: {len(X)}')
print(f'Number of classes: {len(classes)}')
print(f'Feature dimension: {X.shape[-1]}')

## 4. Training Loop (K-Fold CV with per-fold normalization)

In [ ]:
import json
from datetime import datetime
from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset

from src.core.models import GRUModel, MLPModel
from src.train.evaluator import compute_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

input_dim = X.shape[-1]
num_classes = len(classes)

skf = StratifiedKFold(n_splits=CONFIG['k_folds'], shuffle=True, random_state=42)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(y)), y)):
    print(f'\n{"=" * 60}')
    print(f'Fold {fold + 1}/{CONFIG["k_folds"]}')
    print(f'{"=" * 60}')

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # Per-fold normalization (no data leakage)
    mean = X_train.mean(axis=0, keepdims=True)
    std = X_train.std(axis=0, keepdims=True) + 1e-8
    X_train = (X_train - mean) / std
    X_val = (X_val - mean) / std

    # Augmentation (applied after normalization, aware of seq_mode)
    if CONFIG['augment_factor'] > 0:
        X_train, y_train = augment_data(
            X_train, y_train, classes,
            augmentation_factor=CONFIG['augment_factor'],
            noise_level=CONFIG['noise_level'],
            scale_range=CONFIG['scale_range'],
        )

    print(f'Train: {len(X_train)} samples, Val: {len(X_val)} samples')

    # Create model
    if CONFIG['model_type'] == 'gru':
        model = GRUModel(
            input_dim=input_dim,
            hidden_dim=CONFIG['hidden_dim'],
            num_layers=CONFIG['num_layers'],
            num_classes=num_classes,
            dropout=CONFIG['dropout']
        ).to(device)
    else:
        model = MLPModel(
            input_dim=input_dim,
            hidden_dim=CONFIG['hidden_dim'],
            num_layers=CONFIG['num_layers'],
            num_classes=num_classes,
            dropout=CONFIG['dropout']
        ).to(device)

    # DataLoaders
    train_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_train), torch.LongTensor(y_train)),
        batch_size=CONFIG['batch_size'], shuffle=True, num_workers=0
    )
    val_loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_val), torch.LongTensor(y_val)),
        batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0
    )

    # Class weights for imbalance
    class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
    criterion = torch.nn.CrossEntropyLoss(
        weight=torch.FloatTensor(class_weights).to(device)
    )
    optimizer = AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=1e-4)

    # Training
    best_acc = 0.0
    patience_counter = 0
    best_state = None

    for epoch in range(CONFIG['epochs']):
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(Xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            train_correct += (out.argmax(1) == yb).sum().item()
            train_total += yb.size(0)

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(device), yb.to(device)
                out = model(Xb)
                val_loss += criterion(out, yb).item()
                val_correct += (out.argmax(1) == yb).sum().item()
                val_total += yb.size(0)
                all_preds.extend(out.argmax(1).cpu().numpy())
                all_labels.extend(yb.cpu().numpy())

        train_acc = 100.0 * train_correct / train_total
        val_acc = 100.0 * val_correct / val_total

        print(f'  Epoch {epoch+1:3d}/{CONFIG["epochs"]}: '
              f'Train Loss: {train_loss/len(train_loader):.4f}, '
              f'Train Acc: {train_acc:.2f}% | '
              f'Val Loss: {val_loss/len(val_loader):.4f}, '
              f'Val Acc: {val_acc:.2f}%')

        # Early stopping
        if val_acc > best_acc:
            best_acc = val_acc
            patience_counter = 0
            best_state = {
                'model_state_dict': model.state_dict(),
                'epoch': epoch,
                'val_acc': val_acc,
                'fold': fold,
            }
        else:
            patience_counter += 1
            if patience_counter >= CONFIG['patience']:
                print(f'  Early stopping at epoch {epoch + 1}')
                break

    # Evaluate best model
    model.load_state_dict(best_state['model_state_dict'])
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            out = model(Xb)
            probs = torch.softmax(out, dim=1)
            all_preds.extend(out.argmax(1).cpu().numpy())
            all_labels.extend(yb.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    metrics = compute_metrics(
        np.array(all_labels), np.array(all_preds), classes, y_probs=np.array(all_probs)
    )

    result = {
        'fold': fold,
        'best_val_acc': best_acc,
        'val_f1': metrics['f1_score'],
        'val_precision': metrics['precision'],
        'val_recall': metrics['recall'],
        'val_top3': metrics['top3_accuracy'],
        'val_top5': metrics['top5_accuracy'],
        'per_class_metrics': metrics['per_class'],
        'confusion_matrix': metrics['confusion_matrix'],
        'most_confused': metrics['most_confused'],
        'model_state': best_state,
        'normalization_mean': mean.flatten().cpu().numpy().tolist() if hasattr(mean, 'cpu') else mean.flatten().tolist(),
        'normalization_std': std.flatten().cpu().numpy().tolist() if hasattr(std, 'cpu') else std.flatten().tolist(),
    }
    fold_results.append(result)
    print(f'Fold {fold + 1} done. Best val acc: {best_acc:.2f}%, F1: {metrics["f1_score"]:.2f}%')

## 5. Save Best Model

In [ ]:
# Find best fold
best_fold_idx = int(np.argmax([r['best_val_acc'] for r in fold_results]))
best_result = fold_results[best_fold_idx]

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
model_dir = os.path.join(WORK_DIR, 'models')
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, f'tsl51_{CONFIG["model_type"]}_{timestamp}.pt')

checkpoint = {
    'model_state_dict': best_result['model_state']['model_state_dict'],
    'state_dict': best_result['model_state']['model_state_dict'],
    'config': CONFIG,
    'model': CONFIG['model_type'],
    'input_dim': input_dim,
    'num_classes': num_classes,
    'classes': [str(c) for c in classes],
    'labels': [str(c) for c in classes],
    'label_to_idx': {str(c): i for i, c in enumerate(classes)},
    'normalization_mean': best_result['normalization_mean'],
    'normalization_std': best_result['normalization_std'],
    'seq_mode': seq_mode,
    'target_frames': target_frames,
    'accuracy': best_result['best_val_acc'] / 100.0,
    'fold': int(best_fold_idx) + 1,
    'timestamp': timestamp,
}

torch.save(checkpoint, model_path)
print(f'Best model saved to: {model_path}')
print(f'Best val accuracy: {best_result["best_val_acc"]:.2f}%')
print(f'F1 score: {best_result["val_f1"]:.2f}%')
print(f'seq_mode={seq_mode}, target_frames={target_frames}')

## 6. Save Training Results

In [ ]:
results_dir = os.path.join(WORK_DIR, 'results')
os.makedirs(results_dir, exist_ok=True)
results_path = os.path.join(results_dir, f'cv_{timestamp}.json')

cv_accs = [r['best_val_acc'] for r in fold_results]
cv_f1s = [r['val_f1'] for r in fold_results]

results_summary = {
    'config': CONFIG,
    'timestamp': timestamp,
    'num_samples': len(X),
    'num_classes': num_classes,
    'input_dim': input_dim,
    'seq_mode': seq_mode,
    'cv_mean': float(np.mean(cv_accs)),
    'cv_std': float(np.std(cv_accs)),
    'fold_results': [
        {
            'fold': r['fold'] + 1,
            'val_acc': r['best_val_acc'],
            'val_f1': r['val_f1'],
            'val_precision': r['val_precision'],
            'val_recall': r['val_recall'],
            'val_top3': r['val_top3'],
            'val_top5': r['val_top5'],
        }
        for r in fold_results
    ],
    'best_fold': best_fold_idx + 1,
    'model_path': model_path,
}

with open(results_path, 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f'Results saved to: {results_path}')
print(f'CV Accuracy: {results_summary["cv_mean"]:.2f}% +- {results_summary["cv_std"]:.2f}%')

## 7. Training Complete

Proceed to evaluation notebook (03_evaluate.ipynb)